In [1]:
# ニューラルネットワークの推論
# ニューラルネットワークの推論の全体図
import numpy as np

rng = np.random.default_rng()

W1 = rng.random((2, 4))
b1 = rng.random(4)
x = rng.random((10, 2))
h = x @ W1 + b1



In [3]:
# シグモイド関数の実装
def sigmoid(x):
    return 1 / (1 + np.exp(-x))
    
x = rng.random((10, 2))
W1 = rng.random((2, 4))
b1 = rng.random(4)
W2 = rng.random((4, 3))
b2 = rng.random(3)

h = x @ W1 + b1
a = sigmoid(h)
s = a @ W2 + b2

In [ ]:
# レイヤとしてのクラス化と順伝播の実装
# Sigmoidレイヤの実装
import numpy as np

class Sigmoid:
    def __init__(self):
        self.params = []

    def forward(self, x):
        return 1 / (1 + np.exp(-x))

In [5]:
# Affineレイヤの実装
class Affine:
    def __init__(self, W, b):
        self.params = [W, b]

    def forward(self, x):
        W, b = self.params
        out = x @ W + b
        return out

In [8]:
# TwoLayerNetの実装
class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size):
        I, H, O = input_size, hidden_size, output_size

        # 重みとバイアスの初期化
        W1 = rng.random((I, H))
        b1 = rng.random(H)
        W2 = rng.random((H, O))
        b2 = rng.random(O)

        # レイヤの生成
        self.layers = [
            Affine(W1, b1),
            Sigmoid(),
            Affine(W2, b2)
        ]

        # 全ての重みをリストにまとめる
        self.params = []
        for layer in self.layers:
            self.params += layer.params

    def predict(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x

In [9]:
x = rng.random((10, 2))
model = TwoLayerNet(2, 4, 3)
s = model.predict(x)

In [11]:
# 微分と勾配
# チェインルール
# 計算グラフ
# 分岐ノード

In [21]:
# Repeatノード
D, N = 8, 7
x = rng.random((1, D))       # 入力
y = np.repeat(x, N, axis=0)  # forward

dy = rng.random((N, D))                       # 仮の勾配
dx = np.sum(dy, axis=0, keepdims=True)        # backward


In [13]:
# Sumノード
import numpy as np
D, N = 8, 7
x = rng.random((N, D))                         # 入力
y = np.sum(x, axis=0, keepdims=True)           # forward

dy = rng.random((1, D))                        # 仮の勾配
dx = np.repeat(dy, N, axis=0)                  # backward

In [14]:
# MatMulノード
class MatMul:
    def __init__(self, W):
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.x = None

    def forward(self, x):
        W, = self.params
        out = x @ W
        self.x = x
        return out

    def backward(self, dout):
        W, =self.params
        dx = dout @ W.T
        dW = self.x.T @ dout
        self.grads[0][...] = dW
        return  dx

In [16]:
# 勾配の導出と逆伝播の実装
# Sigmoidレイヤ
class Sigmoid:
    def __init__(self):
        self.params, self.grads = [], []
        self.out = None

    def forward(self, x):
        out = 1 / (1 + np.exp(-x))
        self.out = out
        return out

    def backward(self, dout):
        dx = dout * (1.0 - self.out) * self.out
        return dx

In [17]:
# Affineレイヤの実装
class Affine:
    def __init__(self, W, b):
        self.params = [W, b]
        self.grads = [np.zeros_like(W), np.zeros_like(b)]
        self.x = None

    def forward(self, x):
        W, b = self.params
        out = x @ W + b
        self.x = x
        return out

    def backward(self, dout):
        W, b = self.params
        dx = dout @ W.T
        dW = self.x.T @ dout
        db = np.sum(dout, axis=0)

        self.grads[0][...] = dW
        self.grads[1][...] = db
        return dx

In [1]:
# Softmax with Lossレイヤ
# 重みの計算
# SGDの実装
class SGD:
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, params, grads):
        for i in range(len(params)):
            params[i] -= self.lr * grads[i]